In [2]:
import numpy as np
import pandas as pd
import os
import scipy.io.wavfile as wavfile
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tuning import tuner

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC

In [5]:
lengths_to_test = [2000,3000,4000,5000]

# Grid degli iperparametri del RandomForest
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt"],
    # se vuoi, puoi aggiungere:
    # "class_weight": [None, "balanced"],
    # "max_samples": [None, 0.5, 0.8]  # solo se bootstrap=True
}

results = []

print(f"{'Length':<8} | {'Params':<70} | {'Accuracy':<8}")
print("-" * 100)

best_acc = 0
best_config = None

for length in lengths_to_test:
    for rf_params in ParameterGrid(rf_param_grid):
        acc = tuner(target_length=length, **rf_params)

        results.append((length, rf_params, acc))
        print(f"{length:<8} | {str(rf_params):<70} | {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            best_config = {"length": length, **rf_params}

print("-" * 100)
print("\nMigliore configurazione trovata:")
print(f"Length: {best_config['length']}")
for k, v in best_config.items():
    if k == "length":
        continue
    print(f"{k}: {v}")
print(f"Accuracy: {best_acc:.4f}")


Length   | Params                                                                 | Accuracy
----------------------------------------------------------------------------------------------------
2000     | {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50} | 0.7700
2000     | {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} | 0.8133
2000     | {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} | 0.8367
2000     | {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50} | 0.7667
2000     | {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100} | 0.8233
2000     | {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200} | 0.8367
2000     | {'max

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn as sk
import os
import scipy.io.wavfile as wavfile
from scipy.signal import resample

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
#padding before fft
from sklearn.model_selection import GridSearchCV

#PARAMETERS:
max_length = 3000    #Number of samples to trim/pad BEFORE FFT  (also this was tuned)

wavs_dev = ((os.listdir("free-spoken-digit/dev")))
wavlist_dev = sorted(wavs_dev, key=lambda x: int(x.split("_")[0]))
devs = []
lengs = []
X_dev = []
y_dev = []
 

for file in wavlist_dev:
    path = os.path.join("free-spoken-digit/dev", file)      #ex: 0_4.wav

    parts = file.split('_')         #----> 0 , 4.wav
    label = int(parts[1].split('.')[0])     #---> 4 , .wav  and i take 4

    y_dev.append(label)
    
    rate, signal = wavfile.read(path)       #rate = 8000

    if len(signal) > max_length:
        signal = signal[:max_length]
    else:
        pad_width = max_length - len(signal)
        signal = np.pad(signal, (0, pad_width), mode='constant')        #Trim or pad the signal

    
    fft_val = np.abs(np.fft.fft(signal))        #perform fft to have a magnitude spectrum of the signal

    X_dev.append(fft_val)

X_train = pd.DataFrame(X_dev)
y_train = pd.DataFrame(y_dev)


wavs_eval = ((os.listdir("free-spoken-digit/eval")))
wavlist_eval = sorted(wavs_eval, key = lambda x: int(x.split(".")[0]) )
devs=[]
lengs= []
X_eval = []

for file in wavlist_eval:
    path = os.path.join("free-spoken-digit/eval", file)
    parts = int(file.split('.')[0])

    
    (rate, signal) = wavfile.read(path)
    
    if len(signal) > max_length:
        signal = signal[:max_length]
    else:
        pad_width = max_length - len(signal)
        signal = np.pad(signal, (0, pad_width), mode='constant')

    fft_val = np.abs(np.fft.fft(signal))
    X_eval.append(fft_val)     #Does the FFT and resample the signal with 6000 samples

X_test = pd.DataFrame(X_eval)


X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, train_size=0.8, random_state=42)

rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt"],
    # se vuoi, puoi aggiungere:
    # "class_weight": [None, "balanced"],
    # "max_samples": [None, 0.5, 0.8]  # solo se bootstrap=True
}
forest = RandomForestClassifier()
gridsearch = GridSearchCV(forest,rf_param_grid,scoring="accuracy",cv=5)

gridsearch.fit(X_train,y_train)



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklea

,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [None, 10, ...], 'max_features': ['sqrt'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2], ...}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn as sk
import os
import scipy.io.wavfile as wavfile
from scipy.signal import resample

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
#padding before fft

#PARAMETERS:
max_length = 3000    #Number of samples to trim/pad BEFORE FFT  (also this was tuned)

wavs_dev = ((os.listdir("free-spoken-digit/dev")))
wavlist_dev = sorted(wavs_dev, key=lambda x: int(x.split("_")[0]))
devs = []
lengs = []
X_dev = []
y_dev = []
 

for file in wavlist_dev:
    path = os.path.join("free-spoken-digit/dev", file)      #ex: 0_4.wav

    parts = file.split('_')         #----> 0 , 4.wav
    label = int(parts[1].split('.')[0])     #---> 4 , .wav  and i take 4

    y_dev.append(label)
    
    rate, signal = wavfile.read(path)       #rate = 8000

    if len(signal) > max_length:
        signal = signal[:max_length]
    else:
        pad_width = max_length - len(signal)
        signal = np.pad(signal, (0, pad_width), mode='constant')        #Trim or pad the signal

    
    fft_val = np.abs(np.fft.fft(signal))        #perform fft to have a magnitude spectrum of the signal

    X_dev.append(fft_val)

X_train = pd.DataFrame(X_dev)
y_train = pd.DataFrame(y_dev)


wavs_eval = ((os.listdir("free-spoken-digit/eval")))
wavlist_eval = sorted(wavs_eval, key = lambda x: int(x.split(".")[0]) )
devs=[]
lengs= []
X_eval = []

for file in wavlist_eval:
    path = os.path.join("free-spoken-digit/eval", file)
    parts = int(file.split('.')[0])

    
    (rate, signal) = wavfile.read(path)
    
    if len(signal) > max_length:
        signal = signal[:max_length]
    else:
        pad_width = max_length - len(signal)
        signal = np.pad(signal, (0, pad_width), mode='constant')

    fft_val = np.abs(np.fft.fft(signal))
    X_eval.append(fft_val)     #Does the FFT and resample the signal with 6000 samples

X_test = pd.DataFrame(X_eval)


#X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, train_size=0.8, random_state=42)

min_samples_leaf= 1
min_samples_split= 2
n_estimators= 200       #chosen by tuning
max_features = "sqrt"
max_depth= 20
forest = RandomForestClassifier(min_samples_leaf=min_samples_leaf,min_samples_split=min_samples_split,n_estimators=n_estimators,max_features=max_features,max_depth=max_depth)


forest.fit(X_train,y_train)
y_pred= forest.predict(X_test)





/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [12]:
np.linspace(0,len(y_pred)-1,len(y_pred),dtype="int")

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [ ]:
data = {'Id':np.linspace(0,len(y_pred)-1,len(y_pred),dtype="int"),'Predicted':y_pred}
submission = pd.DataFrame(data)
submission.to_csv("submission.csv",index=None)



,Id,Predicted
0,0,3
1,1,9
2,2,3
3,3,5
4,4,6
...,...,...
495,495,1
496,496,7
497,497,1
498,498,7
